In [0]:
%run ./01-ATSConfigs

In [0]:
%run ./03-Endpoints

In [0]:

%run ./04-VectorSearch

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from typing import Iterator

# broadcast_conf = spark.sparkContext.broadcast(f"{conf.catalog}.{conf.db}")
broadcast_conf = f"{conf.catalog}.{conf.db}"

@pandas_udf("array<bigint>")
def profile_search(jd_json_column: Iterator[pd.Series]) -> Iterator[pd.Series]:
    from databricks.vector_seach.cient import VectorSearchClient
    vs_client = VectorSearchClient()
    index = vs_client.get_index(
        index_name=f"{broadcast_conf.value}.candidate_profiles_index"
    ) 
    
    def get_profiles(jd_json):
        ids = index.similarity_search(query_text=jd_json, columns=["id"], num_results=2)
        return [int(data[0]) for data in ids["result"]["data_array"]]

    for x in jd_json_column:
        yield x.apply(get_profiles)



In [0]:
class ProfileSearch:
    def __init__(self):
        spark.conf.set(
            "spark.databricks.delta.changeDataFeed.timestampOutOfRange.enabled", "true"
        )

    def get_start_time(self):
        start_time = (
            spark.sql(
                f""" select execution_time as start_time
                from {conf.jobs_metadata_table_name}
                where job_name = '{conf.jd_profile_job_name}'
                order by execution_time desc
                """
            ).first()
            .asDict()["start_time"]
            .strftime("%Y-%m-%d %H:%M:%S")
        
        )

    def get_end_time(self):
        end_time = (
            spark.sql(
                f"""
                select current_timestamp() as end_time
                """
            ).first()
            .asDict()["end_time"]
            .strftime("%Y-%m-%d %H:%M:%S")
        )

    def get_load_date(self):
        load_date = (
            spark.sql(
                f"""select date_add(last_load_date, 1) as load_date
                                    from {conf.jobs_metadata_table_name}
                                    where job_name='{conf.jd_profile_job_name}'
                                    order by last_load_date desc"""
            )
            .first()
            .asDict()["load_date"]
            .strftime("%Y-%m-%d")
        )

    def update_metadata(self,end_date,load_date):
        spark.sql(
                  f"""
                  insert into {conf.jobs_metadata_table_name}
                  values('{conf.jd_profile_job_name}',{load_date},
                  '{end_date}',
                  'job_execution')
                  """
                  )
    def get_prompt(self):
        from datetime import datetime

        current_year = datetime.now().year
        prompt = f"""
        For the candidate resume below, compare it to the provided job description.
        Provide your output as a JSON object with:
        name (from the resume)

        email (from the resume)

        phone (from the resume)

        fit_score (number from 0 to 10, where 10 is a perfect fit)

        matched_skills (list of skills present in both resume and JD)

        missing_skills (list of required JD skills not found in the resume)

        evaluation (a short, objective summary of the candidate’s fit, mentioning strengths and gaps)

        When comparing skills:
        - Focus the fit score and evaluation primarily on technical and role-specific skills.
        - Do not penalize a candidate for missing foundational technical skills (such as object-oriented programming, data structures, algorithms) or soft skills (such as communication skills, analytical skills, teamwork) if their education, job titles, or work experience clearly imply these skills.
        - Infer such skills from relevant job titles, degrees, leadership, or collaborative work
        - Only list these as missing if there is clear evidence the candidate lacks them or if their experience is too junior to reasonably assume them.

        Return a list of such JSON objects, one per resume.
        """
        return prompt
    
    def match_profile(self):
        from pyspark.sql.functions import explode,expr

        start_date = self.get_start_time()
        end_date = self.get_end_time()
        load_date = self.get_load_date()

        jd_df = (spark.read.option("readChangeFeed",'true')
                 .option('starting_timestamp',start_date)
                 .option('ending_timestamp',end_date)
                 .table({conf.jd_silver_table_name})
                 )
        
        jd_profile_df = (
            jd_df.withColumn("profile_id",explode(profile_search('json_content')))
            .selectExpr("id as jd_id","profile_id")
            .write.mode('overwrite')
            .saveAsTable(f'{conf.jd_profile_table_name}_stg')
            
        )

        prompt = self.get_prompt()
        j_df = spark.read.table({conf.jd_silver_table_name})
        p_df = spark.read.table({conf.profile_silver_table_name})
        stg_df = spark.sql.table(f'{conf.jd_profile_table_name}_stg')

        joined_df =(

            stg_df.alias('stg_df')
            .join(j_df.alias('j_df'), stg_df.jd_id == j_df.id)
            .join(p_df.alias('p_df'),stg_df.profile_id == p_df.id)
            .selectExpr(
                "stg_df.jd_id",
                    "j_df.source as jd_source",
                    "j_df.json_content as jd_extract",
                    "stg_df.profile_id",
                    "p_df.source as profile_source",
                    "p_df.json_content as profile_extract",
            )
        )

        jd_profile_df = (
            joined_df.withColumn('ats_summary',expr(f"""
            ai_query(endpoint => '{conf.llm_endpoint_for_chat}',
                    request => CONCAT('{prompt}',
                                    'resume: ', profile_extract,
                                    'Job Description :',jd_extract)) """),
            ).withColumn('generated_date',expr('current_timestamp()'))
        )

        jd_profile_df.write.mode('append').saveAsTable( conf.jd_profile_table_name)

        self.update_metadata(end_date, load_date)

    def assert_count(self, table_name, expected_count):
        print(f"Validating record counts in {table_name}...", end='')
        actual_count = spark.read.table(f"{conf.catalog}.{conf.db}.{table_name}").count()
        assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}" 
        print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

    def validate(self):
        import time
        start = int(time.time())
        print(f"\nValidating Profile matches for JD...")
        self.assert_count(conf.jd_profile_table_name, 8)
        print(f"Validating Profile matches for JD completed in {int(time.time()) - start} seconds")
